# 3 layer GRU Seq2Seq를 활용한 Num2Text

 숫자 <-> 영어, 숫자 <-> 한국어, 영어 <-> 한국어의 세 언어 쌍에 대해 각각 하나의 모델을 사용한다. 각 모델은 목표 언어 태그를 통해 두 방향의 변환을 모두 수행한다. 인코더의 세 GRU 층에서 얻은 최종 은닉 상태로 디코더의 각 층을 초기화하며, 탐욕 탐색(Greedy Search)으로 출력을 생성한다.

num2txt.txt와 num2kor.txt를 `DATA_DIR`에 지정한 폴더에 두어야 한다. 더 큰 하이퍼파라미터 설정으로 학습할 때는 GPU 사용을 권장한다. 저장된 모델 파일이 있으면 마지막 셀만 실행하여 변환 결과를 확인할 수 있다.

에폭 수, 임베딩 차원, 은닉 상태 차원을 더 작게 설정하여 실험한 결과, 학습이 충분히 이루어지지 않았고 평가 정확도도 낮았다. 성능 향상을 위해 EPOCHS = 25, EMBEDDING_DIM = 64, HIDDEN_DIM = 128로 설정했다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import hashlib
import random

import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Actual path of data folder on Google Drive
DATA_DIR = Path('/content/drive/MyDrive/NLP_Fall26/Project_1_nmt')
MODEL_DIR = Path('.')
BATCH_SIZE = 256
EPOCHS = 25
PATIENCE = 5
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
PAIRS = [('num', 'en'), ('num', 'ko'), ('en', 'ko')]
PAD, BOS, EOS, UNK = range(4)
print('Device:', DEVICE)
print('Data Directory:', DATA_DIR)

Device: cuda
Data Directory: /content/drive/MyDrive/NLP_assignment


In [3]:
# list all the files in DATA_DIR
import os

try:
    print(f"Files in {DATA_DIR}:")
    for file in os.listdir(DATA_DIR):
        print(f"- {file}")
except Exception as e:
    print(f"Error accessing directory: {e}")

Files in /content/drive/MyDrive/NLP_assignment:
- num2kor.txt
- num2txt.txt
- num2text_seq2seq_models.pt
- NLP_assignment01.ipynb
- 아시라프_nmt-wo-attention


## `num2kor.txt`와 `num2txt.txt`를 읽고 숫자 문자열을 기준으로 데이터 분할

In [4]:
def read_pairs(path):
    pairs = {}
    with path.open(encoding='utf-8-sig') as file:
        for line_number, line in enumerate(file, 1):
            number, expression = line.rstrip('\r\n').split('\t')
            if not number.isascii() or not number.isdigit() or not expression.strip():
                raise ValueError(f'Invalid row {line_number} in {path}')
            if number in pairs:
                raise ValueError(f'Duplicate number {number} in {path}')
            pairs[number] = expression
    return pairs

english = read_pairs(DATA_DIR / 'num2txt.txt')
korean = read_pairs(DATA_DIR / 'num2kor.txt')
print(f'English: {len(english):,} | Korean: {len(korean):,} | Shared: {len(english.keys() & korean.keys()):,}')

def split_of(number):
    # The same string always belongs to the same split, across all pairs.
    digest = hashlib.sha256(f'{SEED}:{number}'.encode()).hexdigest()
    bucket = int(digest[:16], 16) % 10000
    return 'train' if bucket < 8000 else 'val' if bucket < 9000 else 'test'

def make_records(first, second):
    rows = {'train': [], 'val': [], 'test': []}
    for number in sorted(english.keys() | korean.keys()):
        texts = {'num': number}
        if number in english:
            texts['en'] = english[number]
        if number in korean:
            texts['ko'] = korean[number]
        if first not in texts or second not in texts:
            continue
        split = rows[split_of(number)]
        split.append((number, first, second, texts[first], texts[second]))
        split.append((number, second, first, texts[second], texts[first]))
    return rows

pair_records = {pair: make_records(*pair) for pair in PAIRS}
for pair, splits in pair_records.items():
    print(pair, {split: len(rows) for split, rows in splits.items()})

English: 100,000 | Korean: 100,000 | Shared: 12,738
('num', 'en') {'train': 160156, 'val': 19800, 'test': 20044}
('num', 'ko') {'train': 160272, 'val': 20176, 'test': 19552}
('en', 'ko') {'train': 20430, 'val': 2486, 'test': 2560}


# 토큰화 및 배치 패딩
숫자 문자열과 한국어는 문자 단위로 토큰화하며, 한국어 표현 내부의 공백도 하나의 토큰으로 처리한다. 영어는 단어 단위로 토큰화하며, `thousands`, `hundreds`와 같은 복수형은 단수형으로 바꾼다. `<to:ko>`와 같은 목표 언어 태그는 모델이 지원하는 두 변환 방향 중 어느 방향을 사용할지 지정한다.

In [5]:
PLURALS = {'thousands': 'thousand', 'hundreds': 'hundred', 'millions': 'million', 'billions': 'billion'}

def tokenize(text, language):
    if language == 'en':
        pieces = [PLURALS.get(word, word) for word in text.strip().lower().split()]
    else:
        pieces = list(text.strip())
    return [f'{language}:{piece}' for piece in pieces]

def source_ids(text, source_lang, target_lang, vocab):
    tokens = [f'<to:{target_lang}>'] + tokenize(text, source_lang)
    return [BOS] + [vocab.get(token, UNK) for token in tokens] + [EOS]

def target_ids(text, language, vocab):
    return [BOS] + [vocab.get(token, UNK) for token in tokenize(text, language)] + [EOS]

def output_text(ids, language, itos):
    pieces = []
    for idx in ids:
        if idx == EOS:
            break
        if idx not in (PAD, BOS):
            prefix, separator, piece = itos[idx].partition(':')
            pieces.append(piece if separator and prefix == language else '<unk>')
    return (' ' if language == 'en' else '').join(pieces)

def make_vocab(pair, training_rows):
    # Include basic number symbols even if a rare token is absent from the split.
    alphabets = {
        'num': list('0123456789'),
        'ko': list('영일이삼사오육칠팔구십백천만억 '),
        'en': ('zero one two three four five six seven eight nine ten eleven twelve thirteen '
               'fourteen fifteen sixteen seventeen eighteen nineteen twenty thirty forty fifty '
               'sixty seventy eighty ninety hundred thousand million billion').split(),
    }
    tokens = {f'{lang}:{piece}' for lang in pair for piece in alphabets[lang]}
    for _, src, tgt, x, y in training_rows:
        tokens.update(tokenize(x, src))
        tokens.update(tokenize(y, tgt))
    itos = ['<pad>', '<bos>', '<eos>', '<unk>'] + [f'<to:{lang}>' for lang in pair]
    return itos + sorted(tokens)

def batch_examples(batch):
    x, y = zip(*batch)
    return (pad_sequence(x, batch_first=True, padding_value=PAD),
            pad_sequence(y, batch_first=True, padding_value=PAD))

def make_loader(rows, vocab, shuffle=False):
    dataset = [(torch.tensor(source_ids(x, src, tgt, vocab)),
                torch.tensor(target_ids(y, tgt, vocab))) for _, src, tgt, x, y in rows]
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=batch_examples)

## GRU encoder and decoder
패킹된 입력 시퀀스를 처리한 인코더는 세 GRU 층의 최종 은닉 상태를 반환한다. 디코더는 이 상태들을 쌓아 초기 은닉 상태로 사용하고, 각 시점에서 어휘에 포함된 모든 토큰의 점수(로짓)를 계산한다.

In [6]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=3):
        super(Encoder, self).__init__()
        if num_layers != 3:
            raise ValueError('This implementation requires three GRU layers (s1, s2, s3).')
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)

    def forward(self, x):
        # x: (batch, seq_len)
        # Create mask for padding
        lengths = (x != 0).sum(dim=1).cpu()
        embedded = self.embedding(x) # (batch, seq_len, embedding_dim)
        # Pack sequence for efficient computation
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False)
        packed_out, hidden = self.gru(packed)
        # Unpack sequence
        output, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=x.size(1))
        return output, hidden[0], hidden[1], hidden[2]

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=3):
        super(Decoder, self).__init__()
        if num_layers != 3:
            raise ValueError('This implementation requires three GRU layers (s1, s2, s3).')
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, s1, s2, s3):
        lengths = (x != 0).sum(dim=1).cpu().clamp(min=1)
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False)

        # Stack hidden states from all layers
        hidden = torch.stack([s1, s2, s3], dim=0)
        packed_out, out_hidden = self.gru(packed, hidden)
        output, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=x.size(1))
        logits = self.fc(output) # (batch, seq_len, vocab_size)
        return logits, out_hidden[0], out_hidden[1], out_hidden[2]

def batch_loss(y_true, y_pred):
    # Flatten token positions, then mask PAD targets.
    batch_size, seq_len, vocab_size = y_pred.shape
    loss_fn = nn.CrossEntropyLoss(reduction='none')
    loss = loss_fn(y_pred.reshape(-1, vocab_size), y_true.reshape(-1))
    loss = loss.reshape(batch_size, seq_len)
    mask = (y_true != 0).float()
    return (loss * mask).sum() / mask.sum().clamp(min=1)

In [7]:
def predict(encoder, decoder, x_batch, y_batch):
    _, s1, s2, s3 = encoder(x_batch)
    logits, _, _, _ = decoder(y_batch, s1, s2, s3)
    return logits

def run_epoch(encoder, decoder, loader, optimizer=None):
    training = optimizer is not None
    encoder.train(training)
    decoder.train(training)
    total_loss, total_tokens = 0.0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            expected = y[:, 1:]
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = predict(encoder, decoder, x, y[:, :-1])
            loss = batch_loss(expected, logits)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(list(encoder.parameters()) + list(decoder.parameters()), 1.0)
                optimizer.step()
            count = int(expected.ne(PAD).sum().item())
            total_loss += loss.item() * count
            total_tokens += count
    return total_loss / total_tokens

def fit_pair(pair, rows):
    itos = make_vocab(pair, rows['train'])
    vocab = {token: i for i, token in enumerate(itos)}
    train_loader = make_loader(rows['train'], vocab, shuffle=True)
    val_loader = make_loader(rows['val'], vocab)
    encoder = Encoder(len(itos), EMBEDDING_DIM, HIDDEN_DIM).to(DEVICE)
    decoder = Decoder(len(itos), EMBEDDING_DIM, HIDDEN_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)
    path = MODEL_DIR / ('num2text_' + '_'.join(pair) + '.pt')
    max_steps = max(len(tokenize(row[4], row[2])) for row in rows['train']) + 6
    best, stale = float('inf'), 0
    for epoch in range(1, EPOCHS + 1):
        train_loss = run_epoch(encoder, decoder, train_loader, optimizer)
        val_loss = run_epoch(encoder, decoder, val_loader)
        print(f'{pair} epoch {epoch:02d}: train={train_loss:.4f}, val={val_loss:.4f}', flush=True)
        if val_loss < best:
            best, stale = val_loss, 0
            torch.save(dict(pair=list(pair), itos=itos, embedding_dim=EMBEDDING_DIM,
                            hidden_dim=HIDDEN_DIM, encoder=encoder.state_dict(),
                            decoder=decoder.state_dict(), epoch=epoch, max_steps=max_steps), path)
        else:
            stale += 1
            if stale >= PATIENCE:
                break
    return path

## Train all three models
디코더는 목표 시퀀스를 한 토큰씩 밀어 다음 토큰을 예측하도록 학습하며, 패딩 토큰을 제외한 교차 엔트로피 손실을 사용한다. 각 언어 쌍에서 검증 손실이 가장 낮은 체크포인트를 저장한다.

In [8]:
paths = {pair: fit_pair(pair, pair_records[pair]) for pair in PAIRS}

('num', 'en') epoch 01: train=1.5013, val=0.7927
('num', 'en') epoch 02: train=0.4228, val=0.2144
('num', 'en') epoch 03: train=0.1311, val=0.0766
('num', 'en') epoch 04: train=0.0526, val=0.0426
('num', 'en') epoch 05: train=0.0265, val=0.0229
('num', 'en') epoch 06: train=0.0206, val=0.0178
('num', 'en') epoch 07: train=0.0138, val=0.0260
('num', 'en') epoch 08: train=0.0118, val=0.0121
('num', 'en') epoch 09: train=0.0101, val=0.0364
('num', 'en') epoch 10: train=0.0080, val=0.0074
('num', 'en') epoch 11: train=0.0081, val=0.0071
('num', 'en') epoch 12: train=0.0056, val=0.0100
('num', 'en') epoch 13: train=0.0062, val=0.0060
('num', 'en') epoch 14: train=0.0063, val=0.0060
('num', 'en') epoch 15: train=0.0035, val=0.0093
('num', 'en') epoch 16: train=0.0054, val=0.0515
('num', 'en') epoch 17: train=0.0042, val=0.0072
('num', 'en') epoch 18: train=0.0036, val=0.0032
('num', 'en') epoch 19: train=0.0034, val=0.0046
('num', 'en') epoch 20: train=0.0035, val=0.0073
('num', 'en') epoch 

## 학습에 사용하지 않은 숫자의 완전 일치 정확도 측정
탐욕 탐색(Greedy Search)은 입력 시퀀스와 모델이 이전 시점까지 생성한 토큰만을 사용한다. 종료 토큰(EOS)을 포함한 모든 토큰이 정답 시퀀스와 일치할 때만 정답으로 처리한다.

In [9]:
def load_pair(path, device):
    state = torch.load(path, map_location=device, weights_only=True)
    args = (len(state['itos']), state['embedding_dim'], state['hidden_dim'])
    encoder, decoder = Encoder(*args).to(device), Decoder(*args).to(device)
    encoder.load_state_dict(state['encoder'])
    decoder.load_state_dict(state['decoder'])
    return encoder.eval(), decoder.eval(), state

@torch.no_grad()
def greedy_decode(encoder, decoder, source, target_lang, itos, max_steps):
    _, s1, s2, s3 = encoder(source)
    token = source.new_full((source.size(0), 1), BOS)
    finished = torch.zeros(source.size(0), dtype=torch.bool, device=source.device)
    allowed = torch.tensor([i == EOS or t.startswith(target_lang + ':')
                            for i, t in enumerate(itos)], device=source.device)
    generated = []
    for _ in range(max_steps):
        logits, s1, s2, s3 = decoder(token, s1, s2, s3)
        scores = logits[:, -1].masked_fill(~allowed, float('-inf'))
        next_id = scores.argmax(dim=-1)
        next_id = next_id.masked_fill(finished, EOS)
        generated.append(next_id)
        finished |= next_id.eq(EOS)
        if finished.all():
            break
        token = next_id.unsqueeze(1)
    return torch.stack(generated, dim=1).cpu().tolist()

def translate(text, src, tgt, bundle):
    encoder, decoder, state = bundle
    if src == tgt or set((src, tgt)) != set(state['pair']):
        raise ValueError('Select the model for this source and target pair.')
    itos = state['itos']
    vocab = {token: i for i, token in enumerate(itos)}
    ids = source_ids(text, src, tgt, vocab)
    if UNK in ids:
        raise ValueError('Input contains tokens outside the saved vocabulary.')
    device = next(encoder.parameters()).device
    source = torch.tensor([ids], device=device)
    generated = greedy_decode(encoder, decoder, source, tgt, itos, state['max_steps'])[0]
    return output_text(generated, tgt, itos)

for pair, path in paths.items():
    encoder, decoder, state = load_pair(path, DEVICE)
    itos = state['itos']
    vocab = {token: i for i, token in enumerate(itos)}
    for src, tgt in (pair, pair[::-1]):
        rows = [row for row in pair_records[pair]['test'] if row[1:3] == (src, tgt)]
        correct, total = 0, 0
        for source, target in make_loader(rows, vocab):
            predictions = greedy_decode(encoder, decoder, source.to(DEVICE), tgt, itos, state['max_steps'])
            for pred, expected in zip(predictions, target.tolist()):
                if EOS in pred:
                    pred = pred[:pred.index(EOS) + 1]
                gold = [i for i in expected[1:] if i != PAD]
                correct += pred == gold
                total += 1
        print(f'{src} -> {tgt}: {correct}/{total} exact matches ({correct/total:.2%})')

num -> en: 9995/10022 exact matches (99.73%)
en -> num: 9908/10022 exact matches (98.86%)
num -> ko: 9770/9776 exact matches (99.94%)
ko -> num: 9774/9776 exact matches (99.98%)
en -> ko: 1260/1280 exact matches (98.44%)
ko -> en: 1263/1280 exact matches (98.67%)


## 저장된 모델을 불러와 여섯 가지 변환 방향 테스트하기
저장된 체크포인트 파일 세 개(num2text_en_ko.pt, num2text_num_en.pt, num2text_num_ko.pt)를 이 노트북과 같은 폴더에 둔다. PyTorch가 설치되어 있으면 이 셀만 실행해도 된다.

사용 방법은 다음과 같다. `convert(text, src, tgt)`에 입력 문자열과 언어 코드(`num`, `en`, `ko`)를 넣는다. 숫자는 정수가 아니라 문자열(`'12345'`)로 넣으며 최대 10자리까지 지원한다. 영어 복수형(`thousands`, `hundreds`)과 띄어쓰기 없는 한글(`일만이천삼백사십오`)도 입력할 수 있다.

영어와 한국어 사이의 변환은 기본적으로 숫자를 거친다(`en -> num -> ko`, `ko -> num -> en`). 두 언어에 공통으로 들어 있는 데이터에는 9~10자리 숫자가 없어서 직접 학습한 `en <-> ko` 모델은 긴 숫자를 본 적이 없기 때문이다. `via_number=False`를 주면 직접 학습한 모델을 사용한다. 다른 입력을 시험하려면 `examples`의 문자열을 수정한다.

In [10]:
import re
from pathlib import Path
import torch
from torch import nn

PAD, BOS, EOS, UNK = range(4)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_DIR = Path('.')

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=3):
        super(Encoder, self).__init__()
        if num_layers != 3:
            raise ValueError('This implementation requires three GRU layers (s1, s2, s3).')
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)

    def forward(self, x):
        lengths = (x != 0).sum(dim=1).cpu()
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False)
        packed_out, hidden = self.gru(packed)
        output, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=x.size(1))
        return output, hidden[0], hidden[1], hidden[2]

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=3):
        super(Decoder, self).__init__()
        if num_layers != 3:
            raise ValueError('This implementation requires three GRU layers (s1, s2, s3).')
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, s1, s2, s3):
        lengths = (x != 0).sum(dim=1).cpu().clamp(min=1)
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False)
        hidden = torch.stack([s1, s2, s3], dim=0)
        packed_out, out_hidden = self.gru(packed, hidden)
        output, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=x.size(1))
        logits = self.fc(output)
        return logits, out_hidden[0], out_hidden[1], out_hidden[2]

PLURALS = {'thousands': 'thousand', 'hundreds': 'hundred', 'millions': 'million', 'billions': 'billion'}

def tokenize(text, language):
    if language == 'en':
        pieces = [PLURALS.get(word, word) for word in text.strip().lower().split()]
    else:
        pieces = list(text.strip())
    return [f'{language}:{piece}' for piece in pieces]

def source_ids(text, source_lang, target_lang, vocab):
    tokens = [f'<to:{target_lang}>'] + tokenize(text, source_lang)
    return [BOS] + [vocab.get(token, UNK) for token in tokens] + [EOS]

def target_ids(text, language, vocab):
    return [BOS] + [vocab.get(token, UNK) for token in tokenize(text, language)] + [EOS]

def output_text(ids, language, itos):
    pieces = []
    for idx in ids:
        if idx == EOS:
            break
        if idx not in (PAD, BOS):
            prefix, separator, piece = itos[idx].partition(':')
            pieces.append(piece if separator and prefix == language else '<unk>')
    return (' ' if language == 'en' else '').join(pieces)

def load_pair(path, device):
    state = torch.load(path, map_location=device, weights_only=True)
    args = (len(state['itos']), state['embedding_dim'], state['hidden_dim'])
    encoder, decoder = Encoder(*args).to(device), Decoder(*args).to(device)
    encoder.load_state_dict(state['encoder'])
    decoder.load_state_dict(state['decoder'])
    return encoder.eval(), decoder.eval(), state

@torch.no_grad()
def greedy_decode(encoder, decoder, source, target_lang, itos, max_steps):
    _, s1, s2, s3 = encoder(source)
    token = source.new_full((source.size(0), 1), BOS)
    finished = torch.zeros(source.size(0), dtype=torch.bool, device=source.device)
    allowed = torch.tensor([i == EOS or t.startswith(target_lang + ':')
                            for i, t in enumerate(itos)], device=source.device)
    generated = []
    for _ in range(max_steps):
        logits, s1, s2, s3 = decoder(token, s1, s2, s3)
        scores = logits[:, -1].masked_fill(~allowed, float('-inf'))
        next_id = scores.argmax(dim=-1)
        next_id = next_id.masked_fill(finished, EOS)
        generated.append(next_id)
        finished |= next_id.eq(EOS)
        if finished.all():
            break
        token = next_id.unsqueeze(1)
    return torch.stack(generated, dim=1).cpu().tolist()

def normalize_ko(text):
    text = text.replace(' ', '')
    if text.startswith('일만'):
        text = text[1:]
    return re.sub(r'([만억])(?=.)', r'\1 ', text)

def translate(text, src, tgt, bundle):
    encoder, decoder, state = bundle
    if src == tgt or set((src, tgt)) != set(state['pair']):
        raise ValueError('Select the model for this source and target pair.')
    if src == 'ko':
        text = normalize_ko(text)
    itos = state['itos']
    vocab = {token: i for i, token in enumerate(itos)}
    ids = source_ids(text, src, tgt, vocab)
    if UNK in ids:
        raise ValueError('Input contains tokens outside the saved vocabulary.')
    device = next(encoder.parameters()).device
    source = torch.tensor([ids], device=device)
    generated = greedy_decode(encoder, decoder, source, tgt, itos, state['max_steps'])[0]
    return output_text(generated, tgt, itos)

PAIRS = [('num', 'en'), ('num', 'ko'), ('en', 'ko')]
bundles = {pair: load_pair(MODEL_DIR / ('num2text_' + '_'.join(pair) + '.pt'), DEVICE) for pair in PAIRS}

def convert(text, src, tgt, via_number=True):
    if via_number and {src, tgt} == {'en', 'ko'}:
        middle = translate(text, src, 'num', bundles[('num', src)])
        return translate(middle, 'num', tgt, bundles[('num', tgt)])
    pair = next(p for p in PAIRS if set(p) == {src, tgt})
    return translate(text, src, tgt, bundles[pair])

en_long = 'nine billion eight hundred seventy six million five hundred forty three thousand two hundred ten'
ko_long = '구십팔억 칠천육백오십사만 삼천이백십'
examples = [
    ('num', 'en', '12345'),
    ('en', 'num', 'twelve thousands three hundreds forty five'),
    ('num', 'ko', '12345'),
    ('ko', 'num', '일만이천삼백사십오'),
    ('en', 'ko', 'twelve thousands three hundreds forty five'),
    ('ko', 'en', '일만이천삼백사십오'),
    ('num', 'en', '9876543210'),
    ('en', 'num', en_long),
    ('num', 'ko', '9876543210'),
    ('ko', 'num', ko_long),
    ('en', 'ko', en_long),
    ('ko', 'en', ko_long),
]
for src, tgt, text in examples:
    print(f'{src} -> {tgt} | {text} -> {convert(text, src, tgt)}')


num -> en | 12345 -> twelve thousand three hundred forty five
en -> num | twelve thousands three hundreds forty five -> 12345
num -> ko | 12345 -> 만 이천삼백사십오
ko -> num | 일만이천삼백사십오 -> 12345
en -> ko | twelve thousands three hundreds forty five -> 만 이천삼백사십오
ko -> en | 일만이천삼백사십오 -> twelve thousand three hundred forty five
num -> en | 9876543210 -> nine billion eight hundred seventy six million five hundred forty three thousand two hundred ten
en -> num | nine billion eight hundred seventy six million five hundred forty three thousand two hundred ten -> 9876543210
num -> ko | 9876543210 -> 구십팔억 칠천육백오십사만 삼천이백십
ko -> num | 구십팔억 칠천육백오십사만 삼천이백십 -> 9876543210
en -> ko | nine billion eight hundred seventy six million five hundred forty three thousand two hundred ten -> 구십팔억 칠천육백오십사만 삼천이백십
ko -> en | 구십팔억 칠천육백오십사만 삼천이백십 -> nine billion eight hundred seventy six million five hundred forty three thousand two hundred ten
